# Workflow Evaluation

Evaluation is critical for understanding how well your agent performs. This tutorial covers how to evaluate workflows using the NAT SDK.

## What You'll Learn

1. Understanding evaluation concepts
2. Creating evaluation datasets
3. Configuring evaluators (metrics)
4. Running evaluations via SDK
5. Running evaluations via CLI
6. Analyzing evaluation results

## Why Evaluate?

- **Measure performance** - Quantify how well your agent answers questions
- **Compare models** - Test different LLMs or prompts
- **Detect regressions** - Ensure changes don't break functionality
- **Benchmark** - Establish baseline metrics for improvement


In [ ]:
import sys
from pathlib import Path

# Setup
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


## Step 1: Create a Workflow to Evaluate

First, let's create a simple calculator workflow that we'll evaluate:


In [ ]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create tools
time_tool = CurrentTimeTool(name="current_time")

# Import calculator if available
try:
    from nat_simple_calculator.register import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
except ImportError:
    tools = [time_tool]
    print("⚠️  Calculator not installed. Install with:")
    print("   uv pip install -e examples/getting_started/simple_calculator")

# Create agent and workflow
agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
)

workflow = NatWorkflow(entrypoint=agent)
print("✅ Workflow created")


## Step 2: Understanding Evaluation Datasets

Evaluation datasets contain:
- **Input**: The question/prompt to send to the agent
- **Expected Output**: The correct answer (ground truth)

NAT supports JSON and CSV dataset formats:

### JSON Format
```json
[
  {
    "input": "What is 2 + 2?",
    "expected_output": "4"
  },
  {
    "input": "What is 10 * 5?",
    "expected_output": "50"
  }
]
```

### CSV Format
```csv
input,expected_output
"What is 2 + 2?","4"
"What is 10 * 5?","50"
```


## Step 3: Create an Evaluation Dataset

Let's create a sample evaluation dataset:


In [ ]:
import json

# Create evaluation dataset
eval_data = [
    {"input": "What is 2 + 2?", "expected_output": "4"},
    {"input": "What is 10 * 5?", "expected_output": "50"},
    {"input": "What is 100 / 4?", "expected_output": "25"},
    {"input": "What is 15 - 7?", "expected_output": "8"},
    {"input": "What is 3 * 3 + 1?", "expected_output": "10"},
]

# Save to file
data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir / "calculator_eval.json"
with open(dataset_path, "w") as f:
    json.dump(eval_data, f, indent=2)

print(f"📊 Evaluation dataset saved to: {dataset_path}")
print(f"   Contains {len(eval_data)} test cases")


## Step 4: Configure Evaluators

Evaluators measure different aspects of your agent's performance. NAT provides several built-in evaluators:

| Evaluator | Measures | Use Case |
|-----------|----------|----------|
| **AnswerAccuracy** | Correctness of answers | General Q&A |
| **AnswerSimilarity** | Semantic similarity to expected | Flexible matching |
| **Faithfulness** | Groundedness in context | RAG applications |
| **ContextRelevancy** | Relevance of retrieved context | RAG applications |


In [ ]:
from nat.eval.rag_evaluator.register import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

# Create an accuracy evaluator
accuracy_evaluator = RagasEvaluator(
    llm=llm,                    # LLM used for evaluation
    metric="AnswerAccuracy",    # Metric to compute
    name="accuracy",            # Name for this evaluator
)

# Configure the evaluation
evaluation = NatEvaluation(
    output_dir=Path("./eval_results"),      # Where to save results
    dataset=EvalDatasetJsonConfig(          # Dataset configuration
        file_path=dataset_path,
    ),
    evaluators=[accuracy_evaluator],        # List of evaluators
)

# Add evaluation to workflow
workflow.add_evaluator(evaluation)

print("✅ Evaluation configured")


## Step 5: Save the Configuration

Export the workflow with evaluation configuration:


In [ ]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "eval_workflow.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")
print("\n" + "=" * 60)
print("GENERATED CONFIGURATION WITH EVALUATION:")
print("=" * 60 + "\n")

with open(config_path) as f:
    print(f.read())


## Step 6: Run Evaluation via Python

Run the evaluation directly in Python:


In [ ]:
# Run evaluation (uncomment to execute)
# await workflow.evaluate()
# print("✅ Evaluation complete! Check ./eval_results for results")


## Step 7: Run Evaluation via CLI

You can also run evaluation from the command line:

```bash
# Basic evaluation
nat eval --config_file configs/eval_workflow.yaml

# Evaluation with custom dataset
nat eval --config_file configs/eval_workflow.yaml \
    --dataset data/calculator_eval.json

# Skip running workflow (use cached results)
nat eval --config_file configs/eval_workflow.yaml --skip_workflow

# Run multiple times for statistical significance
nat eval --config_file configs/eval_workflow.yaml --reps 3
```


## Understanding Evaluation Results

After evaluation, you'll find results in the output directory:

```
eval_results/
├── eval_results.json       # Detailed results for each test case
├── eval_summary.json       # Aggregated metrics
└── eval_log.txt           # Execution log
```

### Example Results
```json
{
  "metrics": {
    "accuracy": {
      "mean": 0.85,
      "std": 0.12,
      "min": 0.6,
      "max": 1.0
    }
  },
  "test_cases": [
    {
      "input": "What is 2 + 2?",
      "expected": "4",
      "actual": "4",
      "accuracy": 1.0
    }
  ]
}
```


## Advanced: Multiple Evaluators

You can use multiple evaluators to get different perspectives:


In [ ]:
# Example: Multiple evaluators
accuracy_eval = RagasEvaluator(llm=llm, metric="AnswerAccuracy", name="accuracy")
similarity_eval = RagasEvaluator(llm=llm, metric="AnswerSimilarity", name="similarity")

multi_evaluation = NatEvaluation(
    output_dir=Path("./eval_results_multi"),
    dataset=EvalDatasetJsonConfig(file_path=dataset_path),
    evaluators=[accuracy_eval, similarity_eval],
)

print("✅ Multi-evaluator configuration created")


## Summary

In this tutorial, you learned:

✅ How evaluation works in NAT  
✅ Creating evaluation datasets (JSON/CSV)  
✅ Configuring evaluators (RagasEvaluator)  
✅ Adding evaluation to workflows  
✅ Running evaluation via SDK and CLI  
✅ Understanding evaluation results  

## Next Steps

- **[10_profiling.ipynb](./10_profiling.ipynb)** - Measure latency and costs
- **[11_optimization.ipynb](./11_optimization.ipynb)** - Improve prompts and parameters
